In [ ]:
import casadi as ca
import cyecca.lie as lie
from cyecca.lie.group_se3 import SE3LieGroupElement

In [ ]:
G = lie.SE3Quat

In [ ]:
M_REF = ca.DM([0.20, 0.05, 0.45]).reshape((3, 1))
G_REF = ca.DM([0.0, 0.0, -9.81]).reshape((3, 1))


In [ ]:
def state_action(X: SE3LieGroupElement, xi: SE3LieGroupElement) -> SE3LieGroupElement:
    A = X.R.to_Matrix()
    a = X.p.param

    R_new = xi.R * X.R
    a_new = A.T @ (xi.p.param + a)

    return G.elem(ca.vertcat(a_new, R_new.param))


def input_action(X: SE3LieGroupElement, ui: ca.SX) -> ca.SX:
    A = X.R.to_Matrix()
    a = X.p.param
    out = A.T @ (ui + a)

    return out


def lift(xi: SE3LieGroupElement, ui: ca.SX) -> ca.SX:
  A = ui - xi.p.param
  b = ca.cross(ui, xi.p.param)
  return ca.vertcat(A, b)


def h(X: SE3LieGroupElement) -> ca.SX:
  return ca.vertcat(X.R.to_Matrix().T @ M_REF, X.R.to_Matrix().T @ G_REF)


def output_action(X: SE3LieGroupElement, xi: SE3LieGroupElement) -> ca.SX:
    AT = xi.R.to_Matrix().T @ X.R.to_Matrix().T

    return ca.vertcat(
        AT @ M_REF,
        AT @ G_REF,
    )
